# Assemble Experiment Artifact
---
Collects the records written by the earlier notebooks into one save_artifact call, reproducing the artifact the frozen notebook wrote from a single kernel.

More information about the package can be found here: [Documentation](https://reionemu.org/)

## Imports

In [1]:
import sys

import numpy as np
import torch

from reionemu import (
    Normalizer,
    read_json,
    save_artifact,
)

## Paths, Configs, and Constants

In [2]:
# Paths
from config import H5_PATH, MODEL_PATH, NORM_PATH, FIG_DIR, REPO_ROOT, make_dirs
from config import ARTIFACT_DIR, ARTIFACT_NAME, RECORDS_DIR
from config import BUILD_INFO_PATH, TRAIN_RESULTS_PATH, KFOLD_SUMMARY_PATH
from config import HELDOUT_METRICS_PATH, HELDOUT_INDICES_PATH

# Package configs
from config import dlcfg, fitcfg, kfcfg

# Constants
from config import SEED, EXPERIMENT_NAME

make_dirs()

PHYSICAL_VALIDATION_PATH = RECORDS_DIR / "physical_validation.json"
MCMC_BATCH_CASES_PATH = RECORDS_DIR / "mcmc_batch_cases.json"

## Environment

In [3]:
print(f"Python:\t\t{sys.version.split()[0]}")
print("-" * 30)
print(f"NumPy:\t\t{np.__version__}")
print(f"PyTorch:\t{torch.__version__}")
print("-" * 30)
print(f"Global Seed:\t{SEED}")

Python:		3.13.1
------------------------------
NumPy:		2.5.2
PyTorch:	2.13.0
------------------------------
Global Seed:	42


## Load Records

A missing one is a hard error rather than a silently omitted artifact section, so the artifact cannot claim to describe a run it has no record of.

In [4]:
required = {
    "Dataset Build": BUILD_INFO_PATH,
    "Training History": TRAIN_RESULTS_PATH,
    "K-Fold Summary": KFOLD_SUMMARY_PATH,
    "Held-Out Metrics": HELDOUT_METRICS_PATH,
    "Held-Out Indices": HELDOUT_INDICES_PATH,
    "Model Checkpoint": MODEL_PATH,
    "Normalizer Mean": NORM_PATH / "X_mean.npy",
    "Normalizer std": NORM_PATH / "X_std.npy",
    "Dataset": H5_PATH,
}
optional = {
    "Physical Validation": PHYSICAL_VALIDATION_PATH,
    "Batch Cases": MCMC_BATCH_CASES_PATH,
}

missing = {name: path for name, path in required.items() if not path.exists()}
if missing:
    raise FileNotFoundError(
        "Missing records:\n"
        + "\n".join(f"  {name}: {path}" for name, path in missing.items())
        + "\n\nRun the notebook that produces each before assembling the artifact."
    )

build_info = read_json(BUILD_INFO_PATH)
train_results = read_json(TRAIN_RESULTS_PATH)
kfold_summary = read_json(KFOLD_SUMMARY_PATH)
heldout_metrics = read_json(HELDOUT_METRICS_PATH)
heldout_indices = np.load(HELDOUT_INDICES_PATH)

physical_validation = (read_json(PHYSICAL_VALIDATION_PATH) if PHYSICAL_VALIDATION_PATH.exists() else None)
mcmc_batch_cases = (read_json(MCMC_BATCH_CASES_PATH) if MCMC_BATCH_CASES_PATH.exists() else None)

ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
modelcfg = ckpt["model_config"]

norm_X = Normalizer(
    mean=np.load(NORM_PATH / "X_mean.npy"),
    std=np.load(NORM_PATH / "X_std.npy"),
)

for name, path in required.items():
    print(f"{name:<20} {path.name}")
for name, path in optional.items():
    print(f"{name:<20} {path.name if path.exists() else '(absent)'}")
print()
print("Architecture:", modelcfg)
print("Held-Out Samples:", len(heldout_indices))

Dataset Build        dataset_build.json
Training History     train_results.json
K-Fold Summary       kfold_summary.json
Held-Out Metrics     heldout_metrics.json
Held-Out Indices     heldout_indices.npy
Model Checkpoint     model.pt
Normalizer Mean      X_mean.npy
Normalizer std       X_std.npy
Dataset              publication_condensed_v6.h5
Physical Validation  physical_validation.json
Batch Cases          mcmc_batch_cases.json

Architecture: {'input_dim': 4, 'output_dim': 5, 'hidden_dim': 20, 'num_hidden_layers': 2, 'activation': 'relu', 'dropout_rate': 0.1}
Held-Out Samples: 200


## Save Artifact

In [5]:
optimizer_config = {
    "name": "AdamW",
    "lr": 1e-3,
    "weight_decay": 1e-5,
}

results_summary = {
    "heldout_mean_percent_error": heldout_metrics["heldout_mean_percent_error"],
    "heldout_coverage_1sigma": heldout_metrics["heldout_coverage_1sigma"],
    "heldout_coverage_2sigma": heldout_metrics["heldout_coverage_2sigma"],
    "heldout_coverage_3sigma": heldout_metrics["heldout_coverage_3sigma"],
    "heldout_reduced_chi2": heldout_metrics["heldout_reduced_chi2"],
}
if physical_validation is not None:
    results_summary["physical_validation"] = {
        "test_set_idx": physical_validation["test_set_idx"],
        "tau_percent_error": physical_validation["tau_percent_error"],
        "xe_mean_percent_error": physical_validation["xe_mean_percent_error"],
    }
if mcmc_batch_cases is not None:
    results_summary["mcmc_batch_cases"] = {
        name: {
            "xe_percent_error_mean": mcmc_batch_cases[name]["xe_percent_error_mean"],
            "xe_percent_error_std": mcmc_batch_cases[name]["xe_percent_error_std"],
        }
        for name in ("random", "central")
    }

artifact = save_artifact(
    ARTIFACT_NAME,
    ARTIFACT_DIR,
    dataset_path=H5_PATH,
    condense_config=build_info["configs"]["condense_config"],
    cl_config=build_info["configs"]["cl_config"],
    build_config=build_info["configs"]["build_config"],
    dataloader_config=dlcfg,
    fit_config=fitcfg,
    kfold_config=kfcfg,
    model_config=modelcfg,
    optimizer_config=optimizer_config,
    results_summary=results_summary,
    metrics=heldout_metrics,
    history={
        "train": train_results,
        "fold_best_val": kfold_summary["fold_best_val"],
        "kfold_mean_val": kfold_summary["mean_best_val"],
        "kfold_std_val": kfold_summary["std_best_val"],
        "final_val_deterministic": kfold_summary["final_val_deterministic"],
        "final_val_mc": kfold_summary["final_val_mc"],
        "kfold_evaluation": kfold_summary["kfold_evaluation"],
        "train_evaluation": kfold_summary["train_evaluation"],
    },
    dataset_prep_stats={
        "condense": build_info["stats"]["condense"],
        "cl": build_info["stats"]["cl"],
        "build_xy": build_info["stats"]["build_xy"],
        "dataset": build_info["dataset"],
        "environment": build_info["environment"],
        "split_info": {
            "heldout_indices": heldout_indices.tolist(),
            "random_seed": SEED,
        },
    },
    normalizers={"X_norm": norm_X},
    checkpoint={
        "model_state_dict": ckpt["model_state_dict"],
        "model_config": modelcfg,
        "normalize_X": ckpt["normalize_X"],
        "normalize_Y": ckpt["normalize_Y"],
    },
    description=(
        f"Experiment using MC dropout model referenced in publication ({EXPERIMENT_NAME}). "
        f"Figures are located in: {FIG_DIR.relative_to(REPO_ROOT)}"
    ),
)

print(f"Artifact:\n{artifact}")

Artifact:
/Users/robertxpearce/Desktop/Research/LEADS Lab/reionemu/reionemu-pasa-2026/artifacts/pearce_2026_reionemu_mc_dropout/mc_dropout_experiment_n_mc_201_epochs_1000_early_stop_150
